# CLI to Python mapping

This notebook maps common `flexgeo2` command-line examples to equivalent Python configuration objects. It is meant as a compact translation guide once you know which analysis you want to run.

In [1]:
from pathlib import Path
import sys
import tempfile


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "pdb2lj5.pdb").is_file():
            return candidate
    raise FileNotFoundError("Could not find the FleXgeo2 repo root containing pyproject.toml and pdb2lj5.pdb")


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    import melodia_py  # noqa: F401
    from flexgeo2 import AnalysisConfig, ClusteringConfig, FlexGeo2App, OutputConfig, ReferenceConfig
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        f"Missing dependency: {exc.name}. Install the project in your notebook kernel with `pip install -e .` "
        "or, if you use uv, run `uv sync` and select the project environment as the Jupyter kernel."
    ) from exc

PDB_FILE = REPO_ROOT / "pdb2lj5.pdb"
tmpdir = tempfile.TemporaryDirectory(prefix="flexgeo2_cli_mapping_")
OUTPUT_DIR = Path(tmpdir.name)
print(f"Temporary output directory: {OUTPUT_DIR}")

Temporary output directory: /var/folders/b7/m1b34cvd23lg7p7rspz3yqnh0000gp/T/flexgeo2_cli_mapping_7n149i01


## Basic run

CLI:

```bash
flexgeo2 pdb2lj5.pdb
```

Python:

In [2]:
basic_config = AnalysisConfig(
    pdb_file=PDB_FILE,
    output=OutputConfig(output_dir=OUTPUT_DIR / "basic"),
)
# result = FlexGeo2App().run(basic_config)

## Chain filtering and worker count

CLI:

```bash
flexgeo2 pdb2lj5.pdb --chain A --n-jobs 4
```

Python:

In [3]:
chain_config = AnalysisConfig(
    pdb_file=PDB_FILE,
    chains=["A"],
    n_jobs=4,
    output=OutputConfig(output_dir=OUTPUT_DIR / "chain_a"),
)

## Verbose outputs

CLI:

```bash
flexgeo2 pdb2lj5.pdb --output-verbose
```

Python:

In [4]:
verbose_config = AnalysisConfig(
    pdb_file=PDB_FILE,
    output=OutputConfig(output_dir=OUTPUT_DIR / "verbose", verbose=True),
)

## Hide individual model traces

CLI:

```bash
flexgeo2 pdb2lj5.pdb --hide-model-traces --max-models-in-plot 20
```

Python:

In [5]:
plot_config = AnalysisConfig(
    pdb_file=PDB_FILE,
    hide_model_traces=True,
    max_models_in_plot=20,
    output=OutputConfig(output_dir=OUTPUT_DIR / "plots"),
)

## Reference model comparison

CLI:

```bash
flexgeo2 pdb2lj5.pdb --reference-model 1
```

Python:

In [6]:
reference_config = AnalysisConfig(
    pdb_file=PDB_FILE,
    reference=ReferenceConfig(model_id="1"),
    output=OutputConfig(output_dir=OUTPUT_DIR / "reference"),
)

For an external reference PDB, the Python equivalent of `--reference-pdb other.pdb --reference-pdb-model 1` is:

In [7]:
external_reference_config = AnalysisConfig(
    pdb_file=PDB_FILE,
    reference=ReferenceConfig(pdb_file="path/to/reference.pdb", pdb_model_id="1"),
    output=OutputConfig(output_dir=OUTPUT_DIR / "external_reference"),
)

## Residue clustering

CLI:

```bash
flexgeo2 pdb2lj5.pdb --cluster-residues --cluster-min-size 8 --cluster-min-samples 4
```

Python:

In [8]:
residue_cluster_config = AnalysisConfig(
    pdb_file=PDB_FILE,
    clustering=ClusteringConfig(
        cluster_residues=True,
        min_cluster_size=8,
        min_samples=4,
    ),
    output=OutputConfig(output_dir=OUTPUT_DIR / "residue_clusters"),
)

## Residue-range clustering

CLI:

```bash
flexgeo2 pdb2lj5.pdb --chain A --cluster-residue-range 45-54 --cluster-residue-range 60-70
```

Python:

In [9]:
range_cluster_config = AnalysisConfig(
    pdb_file=PDB_FILE,
    chains=["A"],
    clustering=ClusteringConfig(cluster_residue_ranges=["45-54", "60-70"]),
    output=OutputConfig(output_dir=OUTPUT_DIR / "range_clusters"),
)

## Running any config

All examples above create config objects. Pass one of them to the app when you want to run that analysis.

In [10]:
# Example: run the smallest example in this notebook.
result = FlexGeo2App().run(
    AnalysisConfig(
        pdb_file=PDB_FILE,
        chains=["A"],
        output=OutputConfig(write_files=False),
    )
)
result.residue_summary_df.head()

,chain,order,name,residue_label,curvature_mean,curvature_std,curvature_min,curvature_max,torsion_mean,torsion_std,torsion_min,torsion_max,models
0,A,1,MET,MET1,0.514353,0.064986,0.325864,0.699961,-0.009660,0.013937,-0.051142,0.036073,301
1,A,2,GLN,GLN2,0.514353,0.064986,0.325864,0.699961,-0.009660,0.013937,-0.051142,0.036073,301
2,A,3,ILE,ILE3,0.583653,0.065199,0.419426,0.815847,-0.085308,0.013672,-0.121378,-0.036480,301
3,A,4,PHE,PHE4,0.616037,0.076530,0.414580,0.939250,-0.120778,0.010640,-0.151191,-0.077521,301
4,A,5,VAL,VAL5,0.834924,0.097212,0.603803,1.123519,-0.103026,0.011403,-0.148288,-0.065921,301
